# CryptoRisk Lab: Bitcoin Downside Risk Simulation

**Research question:** To what extent does a two-state Markov regime-switching model improve 30-day Bitcoin downside-risk estimation compared with a constant-volatility geometric Brownian motion (GBM) baseline?

**Project overview.** CryptoRisk Lab will analyse historical BTC/USDT returns, estimate the probabilities of switching between calmer and more volatile market conditions, simulate alternative 30-day investment outcomes, and compare downside-risk estimates from two models. The intended measures are the probability of exceeding a loss threshold, 95% Value at Risk (VaR), and 95% Expected Shortfall (ES). A later evaluation will compare model outputs with historical observations excluded from calibration.

> **Development status: Day 1 — foundations and model specification.** This notebook currently demonstrates return calculations and a *toy* Markov transition matrix. It does **not** yet fit real Bitcoin data, implement the two complete simulators, or report empirical risk results. Those sections are explicitly marked as planned so that no demonstration is mistaken for evidence.

### Setup and reproducibility

Use Python and common scientific libraries available in free Google Colab. `SEED` is fixed for reproducibility. When external packages are needed later, install them in the first code cell.

In [ ]:
# Day 1 setup: no extra package installation required in Google Colab
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print(f"NumPy {np.__version__}; Pandas {pd.__version__}")
print(f"Random seed: {SEED}")

## 1. Problem and question

Bitcoin returns sometimes exhibit relatively calm periods and sometimes much larger fluctuations. A constant-volatility GBM model treats daily log returns as draws from **one** fixed distribution; a two-state Markov model instead allows the distribution to depend on whether the market is currently in a low- or high-volatility state.

The goal is **not** to forecast a single future Bitcoin price or recommend trades. We will ask whether explicitly modelling changing market conditions materially changes, and potentially improves, estimates of **30-day downside risk**.

**Planned fair comparison**

| Design choice | Decision |
|---|---|
| Asset | Spot BTC/USDT, USDT-denominated (not automatically interchangeable with USD) |
| Calibration data | Daily prices from 2022–2024 |
| Held-out evaluation | Daily prices from 2025, not used for fitting |
| Investment example | 1,000 USDT; adjustable in the final demo |
| Main horizon | 30 days |
| Model A | Constant-volatility GBM |
| Model B | Two-state Markov regime switching |
| Main measures | Loss-threshold exceedance, 95% VaR, 95% ES |
| Numerical experiment | Monte Carlo with recorded seeds, replications and uncertainty intervals |

**Important scientific distinction:** Finding a different estimate is not proof that one model is more accurate. We need verification of our implementation and a clearly limited, separate evaluation against held-out observations.

### Day 1 exercise: simple returns and log returns

For strictly positive daily prices $P_t$ (USDT per BTC), a **simple return** is

$$R_t=\frac{P_t-P_{t-1}}{P_{t-1}},$$

and a **log return** is

$$r_t=\ln\!\left(\frac{P_t}{P_{t-1}}\right)=\ln(1+R_t).$$

Log returns are dimensionless and add over consecutive periods. The next cell uses small **fictional prices for learning only**, not market data.

In [ ]:
def simple_returns(prices):
    """Calculate simple returns from a 1D series of positive daily prices."""
    prices = np.asarray(prices, dtype=float)
    if prices.ndim != 1 or prices.size < 2 or not np.all(np.isfinite(prices)) or np.any(prices <= 0):
        raise ValueError("Expected at least two finite, strictly positive prices")
    return prices[1:] / prices[:-1] - 1.0


def log_returns(prices):
    """Calculate natural-log returns from a 1D series of positive daily prices."""
    prices = np.asarray(prices, dtype=float)
    if prices.ndim != 1 or prices.size < 2 or not np.all(np.isfinite(prices)) or np.any(prices <= 0):
        raise ValueError("Expected at least two finite, strictly positive prices")
    return np.log(prices[1:] / prices[:-1])


# FICTIONAL example only: these are not actual BTC observations.
toy_prices = np.array([40_000.00, 40_800.00, 39_984.00, 41_183.52, 40_360.00, 41_167.20])
toy_simple = simple_returns(toy_prices)
toy_log = log_returns(toy_prices)

toy_results = pd.DataFrame({
    "Day": np.arange(2, toy_prices.size + 1),
    "Close (fictional USDT/BTC)": toy_prices[1:],
    "Simple return (%)": 100 * toy_simple,
    "Log return (%)": 100 * toy_log,
})
display(toy_results.round(4))

print(f"Day 1→2: simple return = {100 * toy_simple[0]:.4f}%; log return = {100 * toy_log[0]:.4f}%")

## 2. Model, assumptions and equations

### Planned modelling assumptions

1. **Daily close data:** All model parameters will be calibrated from one consistently timestamped BTC/USDT daily-price series with documented source, timezone and missing-data checks.
2. **No future information in calibration:** Data from the held-out 2025 evaluation period will not determine 2022–2024 model parameters or regime thresholds.
3. **Two observable volatility regimes:** An initial approach will classify historical days using a trailing volatility estimate, such as a 14-day rolling standard deviation, and a threshold computed on calibration data alone. We will document sensitivity to the window and threshold. Regimes are a modelling convenience, not directly observed physical states.
4. **Conditional return distributions:** Each regime will initially have a separately estimated normal distribution for *daily log returns*. Bitcoin returns may have heavier tails, so normality is a limitation to test and disclose.
5. **First-order Markov transition:** Tomorrow's regime depends probabilistically on today's regime, according to a fitted $2\times2$ transition matrix.
6. **No trading costs or leverage:** Main experiments use buy-and-hold investment value. This is a downside-risk simulator, not a trading system.

### A. Constant-volatility GBM baseline

With daily log-return parameters $\mu$ (mean log return per day) and $\sigma$ (standard deviation of log return per square root of day under the diffusion interpretation), the discrete daily update is

$$V_{t+1}=V_t\exp(\mu+\sigma\varepsilon_t), \qquad \varepsilon_t\sim\mathcal{N}(0,1).$$

For the day-by-day implementation, $\mu$ and $\sigma$ are estimated directly from historical **daily log returns**. They remain fixed throughout a simulated path. $V_t$ is investment value in USDT.

### B. Two-state Markov regime-switching model

Let $Z_t=0$ represent a calm market and $Z_t=1$ a volatile market. The transition matrix is

$$\mathbf{P}=\begin{bmatrix}p_{00}&p_{01}\\p_{10}&p_{11}\end{bmatrix},\quad p_{ij}=\Pr(Z_{t+1}=j\mid Z_t=i),\quad \sum_j p_{ij}=1.$$

Once the next state is drawn, a regime-dependent log return is generated:

$$r_{t+1}\mid Z_{t+1}=j\sim\mathcal{N}(\mu_j,\sigma_j^2),\qquad V_{t+1}=V_t\exp(r_{t+1}).$$

The implementation will define precisely how historical volatility labels align with return observations, so that fitting and simulation use consistent state timing.

### C. Downside risk measures

For initial investment $V_0$, horizon $T=30$ trading/calendar days of daily BTC data, and loss fraction $\ell$, the threshold-exceedance probability is

$$q_\ell=\Pr\bigl(V_T<(1-\ell)V_0\bigr).$$

Define the simulated monetary loss as $L=V_0-V_T$ (USDT). Then

$$\operatorname{VaR}_{0.95}=\text{95th percentile of }L,\qquad
\operatorname{ES}_{0.95}=\mathbb{E}[L\mid L\geq \operatorname{VaR}_{0.95}]$$

for a continuous loss distribution. Monte Carlo estimates will report seeds, repetitions and uncertainty. For example, a 35% loss threshold on 1,000 USDT means a final value **below 650 USDT**.

### Links to the modelling concepts

| Concept | Role in the project |
|---|---|
| Input modelling and random variate generation | Fit return distributions and sample shocks |
| Discrete-time Markov chains | Estimate and simulate transitions between volatility states |
| Monte Carlo integration and the law of large numbers | Estimate loss probabilities and their uncertainty from simulated paths |

### Day 1 exercise: estimating a transition matrix from fictional market states

Suppose we have classified a short fictional sequence of days as `0` (calm) or `1` (volatile). For each consecutive pair, count a transition from current state to next state. Divide each row by its total to obtain transition probabilities.

This exercise is **not** a calibration result. We will replace the fictional sequence with state labels derived from real BTC data.

In [ ]:
# FICTIONAL state sequence, for learning how a Markov matrix is estimated.
toy_states = np.array([0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1], dtype=int)

transition_counts = np.zeros((2, 2), dtype=int)
for current_state, next_state in zip(toy_states[:-1], toy_states[1:]):
    transition_counts[current_state, next_state] += 1

row_totals = transition_counts.sum(axis=1, keepdims=True)
assert np.all(row_totals > 0), "Every state must have an observed outgoing transition"
toy_transition_matrix = transition_counts / row_totals

print("Illustrative transition counts (rows=current, columns=next):")
print(transition_counts)
print("Illustrative transition matrix:")
print(np.round(toy_transition_matrix, 4))

## 3. Implementation

**Implementation roadmap — not yet completed**

- **Data preparation:** Load daily BTC/USDT prices from a documented public dataset; inspect duplicates, missing days, timestamps, and outliers; calculate log returns.
- **GBM:** Fit one daily $\mu$ and $\sigma$ from the calibration sample, then simulate repeated 30-day paths.
- **Regime model:** Calculate trailing volatility, label calm and volatile observations using training-only thresholds, estimate the transition matrix and regime-specific return parameters, then simulate regime and price paths.
- **Risk function:** Estimate $\Pr(V_T<(1-\ell)V_0)$, VaR and ES from terminal outcomes.
- **Reproducibility:** Use explicit random seeds; preserve all assumptions and units; ensure the notebook runs top-to-bottom without manual file paths.

Today's implemented code is limited to the foundational return and transition calculations shown above. The full simulation will be added in subsequent development sessions.

## 4. Verification

**Day 1 checks below verify only the foundational computations.** They must not be presented as verification of the final Bitcoin simulation. Future independent checks will include:

1. GBM Monte Carlo loss probability versus its analytic lognormal formula.
2. A limiting case: identical return distributions in both regimes should match the single-distribution benchmark within Monte Carlo error.
3. Row-sum and range checks for the real transition matrix.
4. Error-scaling study: estimate how Monte Carlo error decreases as simulation count grows, with a log–log plot.
5. A browser-versus-notebook reference test with a stated tolerance.

In [ ]:
# Verification of the *foundational* calculations (fictional data only).
np.testing.assert_allclose(np.exp(toy_log), 1.0 + toy_simple, rtol=1e-12, atol=1e-12)
np.testing.assert_allclose(toy_log.sum(), np.log(toy_prices[-1] / toy_prices[0]), rtol=1e-12, atol=1e-12)
np.testing.assert_allclose(toy_transition_matrix.sum(axis=1), np.ones(2), atol=1e-12)
assert np.all((toy_transition_matrix >= 0) & (toy_transition_matrix <= 1))

print("PASS: log returns and simple returns are mathematically consistent")
print("PASS: log returns add across the complete price interval")
print("PASS: illustrative Markov transition probabilities are valid")
print("NOTE: full GBM and regime-simulation verification has not yet been performed")

## 5. Experiments and results

**No empirical experiment results have been generated yet.** The experiment table below is a *plan*, not evidence.

| Experiment | Comparison / change | Main output |
|---|---|---|
| Primary model comparison | GBM vs two-state Markov model on identical calibration data | 30-day loss probabilities, 95% VaR, 95% ES |
| Threshold sensitivity | 10%, 20%, 35% losses | Estimated event probabilities and confidence intervals |
| Horizon sensitivity | Example: 7, 30, 90 days | How risk changes with horizon |
| Regime sensitivity | Transition persistence and volatility assumptions | Effect on model-estimated downside risk |
| Convergence | Increasing path counts and independent seeds | Error versus $N^{-1/2}$ behavior |
| Held-out evaluation | 2025 observations, without fitting on 2025 data | Limited, transparent out-of-sample risk assessment |

**Planned minimum numerical settings:** 10,000 paths per principal scenario, recorded seed(s), multiple replications as feasible. Monte Carlo uncertainty is not the same as uncertainty about model assumptions. A single 2025 evaluation year may contain few independent extreme-loss events.

## 6. Discussion and limitations

The eventual conclusion must be based on verified experiments. We must not assume beforehand that regime switching performs better merely because volatility changes over time.

**Current limitations (Day 1):** The examples use fictional prices and regime labels. The model is not yet fitted to real Bitcoin observations; no final risk probabilities, comparisons or conclusions have been produced.

**Expected final limitations:** Simplified two-state dynamics, normal conditional returns despite potential heavy tails, sensitivity to regime definitions, crypto market structural changes, USDT-denominated price data, and sparse independent extreme-loss observations in the held-out period.

**Web demo reference result:** To be populated from a verified notebook calculation after both models are implemented. Do **not** invent a reference number or tolerance.

**Next development step:** Retrieve and audit the BTC/USDT daily dataset, then estimate log returns and historical rolling volatility using the calibration period.

## 7. References

These are **starting sources**. Read and verify the publication details and dataset terms before citing them in a completed research report.

1. Binance, "Binance Public Data," GitHub repository. [Online]. Available: https://github.com/binance/binance-public-data (historical BTC/USDT data source).
2. F. Black and M. Scholes, "The pricing of options and corporate liabilities," *Journal of Political Economy*, vol. 81, no. 3, pp. 637–654, 1973, doi: [10.1086/260062](https://doi.org/10.1086/260062) (background on lognormal asset-price modelling).
3. A. M. Law, *Simulation Modeling and Analysis*, 5th ed., McGraw-Hill, 2015 (simulation design and verification background).

Additional peer-reviewed studies on Bitcoin volatility and regime-switching models will be reviewed for the full literature discussion.